# Ejercicio 4

a. Implemente una red con aprendizaje Backpropagation que aprenda la siguiente función:
$$ f(x,y,z) = \sin(x) + \cos(y) + z $$
donde: $x$ e $y \in [0, 2 \pi ]$ y $z \in [− 1,1]$. Para ello construya un conjunto de datos de entrenamiento y un conjunto de evaluación. Muestre la evolución del error de entrenamiento y de evaluación en función de las épocas de entrenamiento.

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(110007)

In [ ]:
class MLP:
    # layers incluye la entrada, por ejemplo (4, 8, 2) son 4 entradas,
    # una capa oculta con 8 neuronas, y 2 neuronas de salida
    def __init__(self, layers, linear_output=False):
        self.weights = []
        self.n_layers = len(layers) - 1
        self.linear_output = linear_output
        for (n_neurons, n_inputs) in zip(layers[1:], layers):
            self.weights.append(np.random.normal(0, 1, size=(n_neurons, n_inputs + 1)))
        self.losses = []
        self.losses_test = []

    def predict(self, X):
        n_p = X.shape[0]
        y = []
        for k in range(n_p):
            Vk = self.forward(X[k])
            y.append(Vk[-1])
        return np.array(y)

    def forward(self, X):
        V = [X]
        for m in range(self.n_layers-1):
            X_ext = np.append(V[m], 1.0)
            Vm = self.activation(np.dot(self.weights[m], X_ext))
            V.append(Vm)

        # última capa
        X_ext = np.append(V[self.n_layers-1], 1.0)
        Vm = np.dot(self.weights[self.n_layers-1], X_ext)
        if not self.linear_output:
            Vm = self.activation(Vm)
        V.append(Vm)
        return V

    def train(self, X, y, X_test=None, y_test=None, epochs=1000, lr=1e-3):
        n_p = X.shape[0]

        for e in range(epochs):
            grads = [np.zeros_like(w) for w in self.weights]
            for k in range(n_p):
                Xk = X[k]
                yk = y[k]
                Vk = self.forward(Xk)
                delta = [None] * self.n_layers
                if self.linear_output:
                    delta[-1] = (yk - Vk[-1])
                else:
                    delta[-1] = (yk - Vk[-1]) * self.activation_dif(Vk[-1])
                for m in range(self.n_layers-1, 0, -1):
                    delta[m-1] = (self.weights[m][:, :-1].T @ delta[m]) * self.activation_dif(Vk[m])
                for m in range(self.n_layers):
                    V_ext = np.append(Vk[m], 1.0)
                    grads[m] += np.outer(delta[m], V_ext)

            for m in range(self.n_layers):
                self.weights[m] += lr * (grads[m] / n_p)

            self.losses.append(self.error(X, y))
            if X_test is not None and y_test is not None:
                self.losses_test.append(self.error(X_test, y_test))

    def error(self, X, y):
        y_pred = self.predict(X).reshape(y.shape)
        return np.mean((y_pred - y) ** 2)

    @staticmethod
    def activation(h):
        return 1 / (1 + np.exp(-h))

    @staticmethod
    def activation_dif(V):
        return V * (1 - V)

In [ ]:
num_samples = 1000
X = np.random.uniform([0, 0, -1], [2*np.pi, 2*np.pi, 1], size=(num_samples, 3))
y = np.sin(X[:,0]) + np.cos(X[:,1]) + X[:,2]

train_fraction = 0.9
Np_train = int(train_fraction*num_samples)
Np_test = num_samples - Np_train

X_train = X[:Np_train,:]
y_train = y[:Np_train]

X_test = X[Np_train:,:]
y_test = y[Np_train:]

In [ ]:
model = MLP(layers=[3, 30, 1], linear_output=True)
model.train(X_train, y_train, X_test=X_test, y_test=y_test, lr=0.1, epochs=1000)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(model.losses, label='Error de entrenamiento')
plt.plot(model.losses_test, label='Error de evaluación')
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
y_pred = model.predict(X_test)

plt.figure(figsize=(4,4))
plt.scatter(y_test, y_pred)
plt.plot([-3, 3], [-3, 3], c='tab:orange')
plt.xlim([-3, 3])
plt.ylim([-3, 3])
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
x = np.linspace(0, np.pi*2, 100)
X = np.c_[x, np.ones(100)*np.pi, np.ones(100)/2]
y = model.predict(X)
plt.plot(x, y, label='Predicción')
plt.plot(x, np.sin(x)+np.cos(np.pi)+0.5, label='Esperada')
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

b) Estudie la evolución de los errores durante el entrenamiento de una red con una capa oculta de 30 neuronas cuando el conjunto de entrenamiento contiene 40 muestras. ¿Que ocurre si el minibatch tiene tamaño 40? ¿Y si tiene tamaño 1?

In [ ]:
np.random.seed(110007)

In [ ]:
num_samples = 60
X = np.random.uniform([0, 0, -1], [2*np.pi, 2*np.pi, 1], size=(num_samples, 3))
y = np.sin(X[:,0]) + np.cos(X[:,1]) + X[:,2]

Np_train = 40
Np_test = num_samples - Np_train

X_train = X[:Np_train,:]
y_train = y[:Np_train]

X_test = X[Np_train:,:]
y_test = y[Np_train:]

In [ ]:
def minibatch_train(model, X_train, y_train, minibatch_size, X_test=None, y_test=None, lr=0.1, epochs=100):
    num_samples = X_train.shape[0] 
    num_batches = int(np.ceil(num_samples / minibatch_size))

    for _ in range(epochs):
        indices = np.random.permutation(num_samples)
        X_shuf = X_train[indices]
        y_shuf = y_train[indices]
        for i in range(num_batches):
            start = minibatch_size * i
            end = start + minibatch_size
            X_batch = X_shuf[start:end]
            y_batch = y_shuf[start:end]
            model.train(X_batch, y_batch, X_test=X_test, y_test=y_test, lr=lr, epochs=1)

In [ ]:
model = MLP(layers=[3, 30, 1], linear_output=True)
minibatch_train(model, X_train, y_train, 40, X_test=X_test, y_test=y_test, lr=0.01, epochs=1000)
#model.train(X_train, y_train, X_test=X_test, y_test=y_test, lr=0.01, epochs=1000)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(model.losses, label='Error de entrenamiento')
plt.plot(model.losses_test, label='Error de evaluación')
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
model = MLP(layers=[3, 30, 1], linear_output=True)
minibatch_train(model, X_train, y_train, 1, X_test=X_test, y_test=y_test, lr=0.01, epochs=1000)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(model.losses, label='Error de entrenamiento')
plt.plot(model.losses_test, label='Error de evaluación')
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()